# Notebook 5 – Ensemble Learning

**Dataset:** Online Retail transactions (`data.csv`)

**Task:** Predict whether an order is from the **United Kingdom** or not, using `Quantity`, `UnitPrice`, `TotalPrice`.

**Intuition:** A single model can be wrong in its own particular way. If we combine several different (or differently-trained) models, their mistakes tend to cancel out, giving a more accurate and stable overall prediction — similar to asking a group of people instead of just one.

## Setup: Load & Split Data

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(3000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1. What is Ensemble Learning?
Combining predictions from **multiple models** into one final prediction, instead of relying on a single model.

In [2]:
from sklearn.tree import DecisionTreeClassifier
single_tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
print("Single tree test acc:", accuracy_score(y_test, single_tree.predict(X_test)))

Single tree test acc: 0.8933333333333333


## 2. Why Ensemble Models?
Individual models each have their own weaknesses (bias or variance). Combining diverse models reduces the impact of any one model's mistakes, usually improving accuracy and stability.

In [3]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print("Ensemble (Random Forest) test acc:", accuracy_score(y_test, forest.predict(X_test)))

Ensemble (Random Forest) test acc: 0.8733333333333333


## 3. Homogeneous Ensembles
All the individual models are the **same type of algorithm** (e.g., many Decision Trees), typically trained on different subsets of data. Random Forest is a classic example.

In [4]:
print("Random Forest = many Decision Trees (homogeneous), combined by voting.")

Random Forest = many Decision Trees (homogeneous), combined by voting.


## 4. Heterogeneous Ensembles
The individual models are **different types of algorithms** (e.g., Logistic Regression + KNN + Decision Tree), each capturing different patterns.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
log_reg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
knn = KNeighborsClassifier(n_neighbors=10).fit(X_train, y_train)
tree = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
print("3 different algorithms trained — ready to combine (heterogeneous ensemble).")

3 different algorithms trained — ready to combine (heterogeneous ensemble).


## 5. Bagging
Short for **Bootstrap Aggregating**: train the same algorithm on many random samples (with replacement) of the data, then average/vote their predictions. Reduces variance (overfitting). Random Forest uses bagging.

In [7]:
from sklearn.ensemble import BaggingClassifier
bagging = BaggingClassifier(DecisionTreeClassifier(random_state=42), n_estimators=50, random_state=42).fit(X_train, y_train)
print("Bagging test acc:", accuracy_score(y_test, bagging.predict(X_test)))

Bagging test acc: 0.875


## 6. Boosting
Trains models **sequentially**, where each new model focuses on fixing the mistakes of the previous ones. Reduces bias, often achieving very high accuracy. AdaBoost and Gradient Boosting are examples.

In [8]:
from sklearn.ensemble import AdaBoostClassifier
boosting = AdaBoostClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)
print("Boosting test acc:", accuracy_score(y_test, boosting.predict(X_test)))

Boosting test acc: 0.8966666666666666


## 7. Voting
Combine predictions from several **different** models by majority vote (classification) or averaging (regression). Simple and effective way to build a heterogeneous ensemble.

In [10]:
from sklearn.ensemble import VotingClassifier
voting = VotingClassifier(estimators=[('lr', log_reg), ('knn', knn), ('dt', tree)], voting='hard')
voting.fit(X_train, y_train)
print("Voting ensemble test acc:", accuracy_score(y_test, voting.predict(X_test)))

Voting ensemble test acc: 0.8933333333333333


## 8. Stacking
Train several base models, then train a **meta-model** on top that learns how to best combine their predictions — more sophisticated than simple voting.

In [11]:
from sklearn.ensemble import StackingClassifier
stacking = StackingClassifier(
    estimators=[('lr', LogisticRegression(max_iter=1000)), ('knn', KNeighborsClassifier(n_neighbors=10))],
    final_estimator=DecisionTreeClassifier(max_depth=3, random_state=42)
)
stacking.fit(X_train, y_train)
print("Stacking ensemble test acc:", accuracy_score(y_test, stacking.predict(X_test)))

Stacking ensemble test acc: 0.8966666666666666


## 9. Blending
Similar to stacking, but the meta-model is trained on a **separate held-out validation set** (rather than using cross-validated predictions like stacking). Simpler to implement, but uses less data for training the base models.

In [12]:
X_tr2, X_blend, y_tr2, y_blend = train_test_split(X_train, y_train, test_size=0.3, random_state=42)
base1 = LogisticRegression(max_iter=1000).fit(X_tr2, y_tr2)
base2 = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr2, y_tr2)
blend_features = np.column_stack([base1.predict(X_blend), base2.predict(X_blend)])
meta_model = LogisticRegression().fit(blend_features, y_blend)
test_blend_features = np.column_stack([base1.predict(X_test), base2.predict(X_test)])
print("Blending test acc:", accuracy_score(y_test, meta_model.predict(test_blend_features)))

Blending test acc: 0.8966666666666666
